# Paso 05 -- Comparar el agente PPO contra la politica base

Corre el agente entrenado (`output/rl_ppo/modelo_ppo.zip`, paso 04) y la
politica base (`asignar_flota`) sobre las MISMAS semillas de evaluacion
(`agente.evaluacion.semillas` en el config -- distintas de
`semilla_entrenamiento`, para no evaluar sobre demandas ya vistas), en la
misma instancia de entrenamiento (escalon chico). Misma semilla -> misma
demanda para las dos politicas (ver `entrenamiento.py`), asi la comparacion
es justa.

**Como se agregan los resultados:** igual que el escalon 3 (semana) -- cada
semilla de evaluacion es su propio episodio independiente;
`metricas.combinar_corridas` junta los N episodios de cada politica en un
solo objeto y `metricas.reporte_completo` da las mismas tablas de siempre,
ahora una por politica, listas para comparar lado a lado. Cero funciones de
metricas nuevas.

**Objetivo de esta primera version:** mostrar si el agente iguala o supera
a la politica base -- no se exige que gane.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt

BASE_DIR = Path.cwd().parent
BB_DIR = (BASE_DIR / "../bergen-boats").resolve()
DEMAND_DIR = (BASE_DIR / "../demand").resolve()
sys.path.insert(0, str(DEMAND_DIR / "src"))
sys.path.insert(0, str(BASE_DIR / "src"))

import masas as demand_masas
import llegadas as demand_llegadas
from entrenamiento import EntornoDemandaAleatoria
from politica_base import asignar_flota
import metricas as met
import visualizacion as viz
from stable_baselines3 import PPO

cfg_sim = yaml.safe_load(open(BASE_DIR / "config" / "instance.yaml", encoding="utf-8"))
cfg_bb = yaml.safe_load(open(BB_DIR / "config" / "instance.yaml", encoding="utf-8"))
cfg_demand = demand_masas.cargar_config(DEMAND_DIR / "config" / "instance.yaml")

resumen_masas = pd.read_csv(DEMAND_DIR / "output" / "masas_por_nodo.csv", index_col="id")
intensidad_od = pd.read_csv(DEMAND_DIR / "output" / "matriz_intensidad_od.csv")
poblacion_total_zonas = resumen_masas["poblacion_total"].sum()
conexiones_fuertes = cfg_bb["garantia"]["conexiones_fuertes"]

nodos = [n["id"] for n in cfg_bb["nodos_demanda"]]
matriz_tiempos = pd.read_csv(BB_DIR / "02_ruteo_navegable" / "output" / "matriz_tiempos_min.csv", index_col=0)

cfg_agente = cfg_sim["agente"]
cfg_entren = cfg_agente["entrenamiento"]
cfg_escalon = cfg_sim["escalones"][cfg_entren["escalon_base"]]
hora_ini_min, hora_fin_min = cfg_escalon["horas"][0] * 60, cfg_escalon["horas"][1] * 60
semillas_eval = cfg_agente["evaluacion"]["semillas"]

# Mismos overrides de recompensa que uso el entrenamiento (04_entrenamiento_rl.ipynb) --
# ver config/instance.yaml, agente.entrenamiento.recompensa_overrides. Se aplican a AMBAS
# politicas aca (agente y base) para que el reward reportado sea comparable 1:1 -- si el
# agente usara una formula de recompensa distinta a la base, la comparacion de reward
# total no significaria nada.
cfg_recompensa_rl = dict(cfg_sim["recompensa"])
cfg_recompensa_rl.update(cfg_entren.get("recompensa_overrides", {}))

RL_DIR = BASE_DIR / "output" / "rl_ppo"
OUT_DIR = BASE_DIR / "output" / "comparacion"
OUT_DIR.mkdir(parents=True, exist_ok=True)

model = PPO.load(str(RL_DIR / "modelo_ppo"))
print("Modelo cargado de", RL_DIR / "modelo_ppo.zip")
print(f"Instancia de evaluacion: {cfg_entren['escalon_base']}, semillas: {semillas_eval}")
print(f"Overrides de recompensa (solo RL): {cfg_entren.get('recompensa_overrides', {})}")


def construir_entorno_eval():
    return EntornoDemandaAleatoria(
        generar_llegadas_dia_fn=demand_llegadas.generar_llegadas_dia,
        cfg_demand=cfg_demand, intensidad_od=intensidad_od, conexiones_fuertes=conexiones_fuertes,
        poblacion_total_zonas=poblacion_total_zonas, horas=cfg_escalon["horas"],
        porcentaje_poblacion_dia=cfg_escalon["porcentaje_poblacion_dia"],
        matriz_tiempos=matriz_tiempos, nodos=nodos, num_barcos=cfg_escalon["num_barcos"],
        capacidad_barco=cfg_bb["flota"]["capacidad_pasajeros"], nodo_inicial=cfg_bb["flota"]["nodo_inicial"],
        paso_tiempo_min=cfg_sim["paso_tiempo_min"], hora_inicio_min=hora_ini_min, hora_fin_min=hora_fin_min,
        cfg_recompensa=cfg_recompensa_rl, unidad_demanda=cfg_sim["unidad_demanda"],
    )


# B.6: consistencia train/eval -- prueba estructural, no solo de nombre. Si 04 y 05
# usaran escalones distintos (p.ej. entrenar en escalon_1, evaluar en escalon_2), la
# dimension del vector aplanado casi seguro seria distinta (depende de num_barcos) --
# esto lo confirma en tiempo de ejecucion, no solo comparando la clave de config.
_dim_modelo = model.observation_space.shape
_dim_eval = construir_entorno_eval().observation_space.shape
assert _dim_modelo == _dim_eval, (
    f"Desajuste train/eval: el modelo espera observaciones de forma {_dim_modelo}, "
    f"pero {cfg_entren['escalon_base']} en esta evaluacion da {_dim_eval} -- entrenar y "
    f"evaluar deben usar el mismo escalon."
)
print(f"OK: train y eval usan el mismo escalon ({cfg_entren['escalon_base']}), "
      f"vector aplanado de forma {_dim_eval} en ambos.")

# VecNormalize (B.2, ver simulacion/README.md diagnostico): si 04_entrenamiento_rl.ipynb
# entreno con esto activo, hay que evaluar con las MISMAS estadisticas (congeladas, no
# se siguen actualizando) -- si no, el agente veria observaciones en una escala distinta
# a la que aprendio. `usar_vecnormalize_eval` se decide por la EXISTENCIA del archivo
# guardado, no leyendo el config (asi sigue funcionando si se evalua un modelo viejo
# entrenado sin esto).
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

VECNORMALIZE_PATH = RL_DIR / "vecnormalize.pkl"
usar_vecnormalize_eval = VECNORMALIZE_PATH.exists()
print(f"VecNormalize en evaluacion: {usar_vecnormalize_eval} "
      f"({'encontrado' if usar_vecnormalize_eval else 'no encontrado'} {VECNORMALIZE_PATH.name})")


Modelo cargado de C:\Users\hfons\Andes\OneDrive - Universidad de los Andes\NHH\NHH-Schedule-Free-Autonomous-Boats-in-Bergen\simulacion\output\rl_ppo\modelo_ppo.zip
Instancia de evaluacion: escalon_1, semillas: [1001, 1002, 1003, 1004, 1005]
Overrides de recompensa (solo RL): {'premio_por_persona_entregada': 0.5, 'peso_movimiento': 0.0}
OK: train y eval usan el mismo escalon (escalon_1), vector aplanado de forma (50,) en ambos.
VecNormalize en evaluacion: True (encontrado vecnormalize.pkl)


## Correr las dos politicas sobre las mismas semillas de evaluacion

In [2]:
def correr_agente(semilla):
    if not usar_vecnormalize_eval:
        env = construir_entorno_eval()
        obs, info = env.reset(seed=semilla)
        reward_total = 0.0
        while True:
            accion, _ = model.predict(obs, deterministic=True)
            obs, r, terminated, truncated, info = env.step(accion)
            reward_total += r
            if truncated or terminated:
                break
        return env, reward_total

    # Con VecNormalize: se steppea el entorno CRUDO directamente (NO a traves de
    # `venv.step()`) y solo se usan las estadisticas de VecNormalize para normalizar
    # la observacion antes de `predict()`. Se intento primero step-ear a traves del
    # DummyVecEnv (mas "de manual" de SB3) pero eso tiene un efecto secundario real:
    # `DummyVecEnv.step_wait()` AUTO-RESETEA el entorno interno en el mismo `step()`
    # en que `done=True` (comportamiento estandar de VecEnv, para no perder un paso
    # al encadenar episodios) -- eso borraba `atendidas_historico`/`log_eventos`
    # ANTES de poder leerlos, y `verificar_conservacion` daba todo en 0. Con el
    # entorno crudo (mismo patron ya usado y verificado en el diagnostico, ver
    # bitacora) el `reset()` lo controlamos nosotros, una sola vez, al principio.
    venv_stats = VecNormalize.load(str(VECNORMALIZE_PATH), DummyVecEnv([construir_entorno_eval]))
    venv_stats.training = False
    venv_stats.norm_reward = False  # comparar sobre recompensa CRUDA, igual que la politica base

    env = construir_entorno_eval()
    obs, info = env.reset(seed=semilla)
    reward_total = 0.0
    while True:
        obs_norm = venv_stats.normalize_obs(obs)
        accion, _ = model.predict(obs_norm, deterministic=True)
        obs, r, terminated, truncated, info = env.step(accion)
        reward_total += r
        if truncated or terminated:
            break
    return env, reward_total


def correr_base(semilla):
    env = construir_entorno_eval()
    obs, info = env.reset(seed=semilla)
    reward_total = 0.0
    while True:
        estado = info["_estado_obj"]
        libres = [b for b in estado.barcos if b.libre]
        decisiones = asignar_flota(libres, estado, matriz_tiempos, env.capacidad_barco, cfg_recompensa_rl)
        accion = np.array([
            env.codificar_accion_barco(decisiones[b.id]) if b.libre else 0
            for b in estado.barcos
        ])
        obs, r, terminated, truncated, info = env.step(accion)
        reward_total += r
        if truncated or terminated:
            break
    return env, reward_total


envs_agente, envs_base = [], []
filas_reward = []
for semilla in semillas_eval:
    env_a, r_a = correr_agente(semilla)
    env_b, r_b = correr_base(semilla)
    envs_agente.append(env_a)
    envs_base.append(env_b)
    filas_reward.append({"semilla": semilla, "reward_agente": r_a, "reward_base": r_b, "diferencia": r_a - r_b})
    print(f"semilla {semilla}: agente={r_a:8.2f}  base={r_b:8.2f}  diferencia={r_a - r_b:+8.2f}")

df_reward = pd.DataFrame(filas_reward)
df_reward.to_csv(OUT_DIR / "reward_por_semilla.csv", index=False)
print(f"\nPromedio: agente={df_reward['reward_agente'].mean():.2f}  base={df_reward['reward_base'].mean():.2f}")


semilla 1001: agente= -537.68  base=  -56.30  diferencia= -481.37


semilla 1002: agente= -976.56  base= -368.46  diferencia= -608.11
semilla 1003: agente= -705.63  base= -174.83  diferencia= -530.81


semilla 1004: agente=-1795.32  base=-1377.23  diferencia= -418.09


semilla 1005: agente= -791.81  base=  -32.45  diferencia= -759.36

Promedio: agente=-961.40  base=-401.85


## Combinar las 5 corridas de cada politica y sacar las metricas de siempre

Mismo mecanismo que el escalon 3 (`metricas.combinar_corridas` +
`metricas.reporte_completo`) -- aca cada "dia" es una semilla de
evaluacion distinta, en vez de un dia de la semana.

In [3]:
corrida_agente = met.combinar_corridas(envs_agente)
corrida_base = met.combinar_corridas(envs_base)

reporte_agente = met.reporte_completo(corrida_agente)
reporte_base = met.reporte_completo(corrida_base)

print("Conservacion agente:", reporte_agente["conservacion"]["cuadra"])
print("Conservacion base:  ", reporte_base["conservacion"]["cuadra"])


Conservacion agente: True
Conservacion base:   True


## Tabla comparativa (globales)

In [4]:
claves_comparar = [
    "unidades_generadas", "unidades_atendidas", "unidades_sin_atender_al_final",
    "pct_atendidas", "espera_media_min", "sistema_medio_min", "sistema_maximo_min",
]
tabla_comparativa = pd.DataFrame({
    "metrica": claves_comparar,
    "agente_ppo": [reporte_agente["globales"][k] for k in claves_comparar],
    "politica_base": [reporte_base["globales"][k] for k in claves_comparar],
})
tabla_comparativa["diferencia"] = tabla_comparativa["agente_ppo"] - tabla_comparativa["politica_base"]

# Recompensa total y movimientos/ocupacion de flota, tambien comparados
tabla_comparativa = pd.concat([tabla_comparativa, pd.DataFrame([
    {"metrica": "reward_total", "agente_ppo": df_reward["reward_agente"].sum(),
     "politica_base": df_reward["reward_base"].sum(),
     "diferencia": df_reward["reward_agente"].sum() - df_reward["reward_base"].sum()},
    {"metrica": "movimientos_totales", "agente_ppo": reporte_agente["por_barco"]["movimientos"].sum(),
     "politica_base": reporte_base["por_barco"]["movimientos"].sum(),
     "diferencia": reporte_agente["por_barco"]["movimientos"].sum() - reporte_base["por_barco"]["movimientos"].sum()},
    {"metrica": "ocupacion_media", "agente_ppo": reporte_agente["por_barco"]["ocupacion_media"].mean(),
     "politica_base": reporte_base["por_barco"]["ocupacion_media"].mean(),
     "diferencia": reporte_agente["por_barco"]["ocupacion_media"].mean() - reporte_base["por_barco"]["ocupacion_media"].mean()},
])], ignore_index=True)

tabla_comparativa.to_csv(OUT_DIR / "tabla_comparativa.csv", index=False)
display(tabla_comparativa.round(2))


,metrica,agente_ppo,politica_base,diferencia
0,unidades_generadas,850.00,850.00,0.00
1,unidades_atendidas,656.00,748.00,-92.00
2,unidades_sin_atender_al_final,194.00,102.00,92.00
3,pct_atendidas,77.18,88.00,-10.82
4,espera_media_min,20.46,16.42,4.04
5,sistema_medio_min,30.42,26.05,4.37
6,sistema_maximo_min,117.13,69.74,47.39
7,reward_total,-4807.00,-2009.26,-2797.74
8,movimientos_totales,177.00,155.00,22.00
9,ocupacion_media,2.97,3.50,-0.52


## Comparacion por par origen-destino

In [5]:
par_agente = reporte_agente["por_par"][["par", "pct_atendidas", "espera_media_min", "sistema_medio_min"]]
par_base = reporte_base["por_par"][["par", "pct_atendidas", "espera_media_min", "sistema_medio_min"]]
por_par_comparado = par_agente.merge(par_base, on="par", suffixes=("_agente", "_base"))
por_par_comparado.to_csv(OUT_DIR / "por_par_comparado.csv", index=False)
display(por_par_comparado.round(1))


,par,pct_atendidas_agente,espera_media_min_agente,sistema_medio_min_agente,pct_atendidas_base,espera_media_min_base,sistema_medio_min_base
0,kleppesto->laksevag,100.0,15.7,21.7,100.0,7.7,13.7
1,kleppesto->bryggen,77.6,18.4,28.0,80.8,17.0,24.9
2,kleppesto->sandviken,29.5,37.0,39.0,100.0,15.6,23.3
3,laksevag->kleppesto,100.0,1.3,50.6,100.0,25.3,58.8
4,laksevag->bryggen,74.5,19.9,27.5,85.7,23.1,28.3
5,laksevag->sandviken,60.0,107.1,117.1,60.0,27.1,37.1
6,bryggen->kleppesto,100.0,17.2,35.9,100.0,17.2,23.9
7,bryggen->laksevag,55.0,22.1,32.6,100.0,7.2,23.1
8,bryggen->sandviken,77.8,6.4,19.0,100.0,11.1,22.7
9,sandviken->kleppesto,100.0,20.3,30.3,100.0,16.2,26.2


## Graficas comparativas

In [6]:
from plotly.subplots import make_subplots

fig_a = viz.graficar_heatmap_cumplimiento(corrida_agente)
fig_b = viz.graficar_heatmap_cumplimiento(corrida_base)

fig = make_subplots(rows=1, cols=2, subplot_titles=["PPO agent -- % served by pair", "Base policy -- % served by pair"])
fig.add_trace(fig_a.data[0], row=1, col=1)
fig.add_trace(fig_b.data[0], row=1, col=2)
fig.update_xaxes(title_text="Destination", row=1, col=1)
fig.update_xaxes(title_text="Destination", row=1, col=2)
fig.update_yaxes(title_text="Origin", row=1, col=1)
fig.update_layout(title="Agent vs. base policy -- % served by O-D pair", template="plotly_white")
fig.write_html(OUT_DIR / "heatmaps_comparados.html")
fig.show()


In [7]:
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Bar(x=[str(s) for s in semillas_eval], y=df_reward["reward_agente"], name="PPO agent", marker_color="#2980b9"))
fig.add_trace(go.Bar(x=[str(s) for s in semillas_eval], y=df_reward["reward_base"], name="base policy", marker_color="#e67e22"))
fig.update_layout(
    title="Evaluation-episode reward -- agent vs. base policy",
    xaxis_title="Evaluation seed", yaxis_title="Episode total reward",
    barmode="group", template="plotly_white",
)
fig.write_html(OUT_DIR / "reward_por_semilla.html")
fig.show()
